# Car Type Classification using Deep Learning
### CS4252 - Sudarshan Phade | CS4259 - Rishikesh Shelhalkar
**Theme:** Vehicle Type Classification  
**Classes:** Hatchback, Sedan, Sports_Car, SUV

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')

## Step 2: Set Project Paths

In [ ]:
import os

# ── Change this to match your Google Drive folder name ──────────────────────
PROJECT = 'CS4252_Sudarshan_Phade_CS4259_Rishikesh_Shelhalkar_DeepLearning_Project'
# ────────────────────────────────────────────────────────────────────────────

BASE         = f'/content/drive/MyDrive/{PROJECT}'
DATASET_DIR  = os.path.join(BASE, 'Dataset Folder')
SOURCE_DIR   = DATASET_DIR
TRAIN_DIR    = os.path.join(DATASET_DIR, 'Training')
VAL_DIR      = os.path.join(DATASET_DIR, 'Validation')
TEST_DIR     = os.path.join(DATASET_DIR, 'Testing')
MODEL_DIR    = os.path.join(BASE, 'Trained Model')
OUTPUT_DIR   = os.path.join(BASE, 'Sample Output')

# Create output directories
for d in [
    TRAIN_DIR, VAL_DIR, TEST_DIR,
    os.path.join(TRAIN_DIR, 'Hatchback'), os.path.join(TRAIN_DIR, 'Sedan'),
    os.path.join(TRAIN_DIR, 'Sports_Car'), os.path.join(TRAIN_DIR, 'SUV'),
    os.path.join(VAL_DIR,   'Hatchback'), os.path.join(VAL_DIR,   'Sedan'),
    os.path.join(VAL_DIR,   'Sports_Car'), os.path.join(VAL_DIR,   'SUV'),
    os.path.join(TEST_DIR,  'Hatchback'), os.path.join(TEST_DIR,  'Sedan'),
    os.path.join(TEST_DIR,  'Sports_Car'), os.path.join(TEST_DIR,  'SUV'),
    MODEL_DIR,
    os.path.join(OUTPUT_DIR, 'training_screenshots'),
    os.path.join(OUTPUT_DIR, 'accuracy_loss_graphs'),
    os.path.join(OUTPUT_DIR, 'confusion_matrix'),
    os.path.join(OUTPUT_DIR, 'sample_predictions'),
]:
    os.makedirs(d, exist_ok=True)

print('All paths set and directories created!')
print('Project root:', BASE)

## Step 3: Import Libraries

In [ ]:
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## Step 4: Split Dataset (70% Train / 15% Validation / 15% Test)

In [ ]:
CLASSES     = ['Hatchback', 'Sedan', 'Sports_Car', 'SUV']
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

random.seed(42)

def split_dataset():
    for cls in CLASSES:
        src = os.path.join(SOURCE_DIR, cls)
        if not os.path.exists(src):
            print(f'WARNING: {src} not found, skipping...')
            continue

        images = sorted([f for f in os.listdir(src)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        random.shuffle(images)

        n       = len(images)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)

        splits = {
            TRAIN_DIR : images[:n_train],
            VAL_DIR   : images[n_train:n_train + n_val],
            TEST_DIR  : images[n_train + n_val:]
        }

        for dest_root, files in splits.items():
            dest = os.path.join(dest_root, cls)
            for f in files:
                dst_file = os.path.join(dest, f)
                if not os.path.exists(dst_file):
                    shutil.copy(os.path.join(src, f), dst_file)

        print(f'{cls:12s} → Train: {len(splits[TRAIN_DIR])},  Val: {len(splits[VAL_DIR])},  Test: {len(splits[TEST_DIR])}')

split_dataset()
print('\nDataset split complete!')

## Step 5: Preprocessing & Data Augmentation

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=42
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print('Classes:', CLASS_NAMES)
print('Train samples:', train_gen.samples)
print('Val samples  :', val_gen.samples)
print('Test samples :', test_gen.samples)

### Visualize Sample Training Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Sample Training Images (after augmentation)', fontsize=14)

batch_imgs, batch_labels = next(train_gen)
for i, ax in enumerate(axes.flat):
    ax.imshow(batch_imgs[i])
    ax.set_title(CLASS_NAMES[np.argmax(batch_labels[i])])
    ax.axis('off')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'training_screenshots', 'sample_augmented_images.png')
plt.savefig(save_path, dpi=150)
plt.show()
print('Saved:', save_path)

## Step 6: Build Model (MobileNetV2 Transfer Learning)

In [ ]:
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 7: Train Phase 1 (Frozen Base)

In [ ]:
MODEL_PATH = os.path.join(MODEL_DIR, 'car_classifier.keras')

callbacks = [
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='val_accuracy', verbose=1),
    EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=4, monitor='val_loss', verbose=1)
]

history1 = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks
)

## Step 8: Train Phase 2 (Fine-Tuning)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks
)

print('Best model saved to:', MODEL_PATH)

## Step 9: Accuracy & Loss Graphs

In [ ]:
def merge(h1, h2, key):
    return h1.history[key] + h2.history[key]

acc      = merge(history1, history2, 'accuracy')
val_acc  = merge(history1, history2, 'val_accuracy')
loss     = merge(history1, history2, 'loss')
val_loss = merge(history1, history2, 'val_loss')
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc,     'b-o', label='Train Accuracy', markersize=4)
ax1.plot(epochs, val_acc, 'r-o', label='Val Accuracy',   markersize=4)
ax1.axvline(len(history1.history['accuracy']), color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, loss,     'b-o', label='Train Loss', markersize=4)
ax2.plot(epochs, val_loss, 'r-o', label='Val Loss',   markersize=4)
ax2.axvline(len(history1.history['loss']), color='gray', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'accuracy_loss_graphs', 'accuracy_loss.png')
plt.savefig(save_path, dpi=150)
plt.show()
print('Saved:', save_path)

## Step 10: Evaluate on Test Set

In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH)

test_loss, test_acc = best_model.evaluate(test_gen, verbose=1)
print(f'\nTest Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

## Step 11: Confusion Matrix

In [ ]:
test_gen.reset()
y_pred_probs = best_model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'confusion_matrix', 'confusion_matrix.png')
plt.savefig(save_path, dpi=150)
plt.show()
print('Saved:', save_path)

## Step 12: Classification Report (Accuracy, Precision, Recall, F1)

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print(report)

report_path = os.path.join(OUTPUT_DIR, 'confusion_matrix', 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(report)
print('Saved:', report_path)

## Step 13: Sample Prediction Results

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Prediction Results', fontsize=15)

sample_images = []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(TEST_DIR, cls)
    imgs = sorted(os.listdir(cls_dir))
    if len(imgs) >= 1:
        sample_images.append((os.path.join(cls_dir, imgs[0]), cls))
    if len(imgs) >= 2:
        sample_images.append((os.path.join(cls_dir, imgs[1]), cls))

for i, ax in enumerate(axes.flat):
    if i >= len(sample_images):
        ax.axis('off')
        continue
    img_path, true_label = sample_images[i]
    img_pil = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    img_arr = tf.keras.preprocessing.image.img_to_array(img_pil) / 255.0
    preds   = best_model.predict(np.expand_dims(img_arr, axis=0), verbose=0)
    pred_label = CLASS_NAMES[np.argmax(preds)]
    confidence = np.max(preds) * 100
    color = 'green' if pred_label == true_label else 'red'
    ax.imshow(mpimg.imread(img_path))
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)', color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'sample_predictions', 'sample_predictions.png')
plt.savefig(save_path, dpi=150)
plt.show()
print('Saved:', save_path)

## Step 14: Final Results Summary

In [ ]:
print('=' * 55)
print('           FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  Test Accuracy  : {test_acc*100:.2f}%')
print(f'  Test Loss      : {test_loss:.4f}')
print(f'  Classes        : {CLASS_NAMES}')
print(f'  Model saved at : {MODEL_PATH}')
print('=' * 55)

## Step 15: Observations & Possible Improvements

### Observations
- The model uses **MobileNetV2** pretrained on ImageNet with two-phase training (frozen base → fine-tuning), which helps achieve good accuracy even with a small dataset.
- **Data augmentation** (rotation, flip, zoom, shift) helps reduce overfitting on the small training set.
- Classes like **Sports_Car** and **Hatchback** may be harder to distinguish due to visual similarity in some angles.
- The confusion matrix reveals which class pairs are most commonly misclassified.

### Possible Improvements
1. **More data** — Collect 150–200 images per class for better generalization.
2. **Stronger augmentation** — Add brightness/contrast jitter and cutout augmentation.
3. **Larger backbone** — Try EfficientNetB0 or ResNet50 for potentially higher accuracy.
4. **Class balancing** — Use class weights to handle the slight imbalance (Sedan/SUV have 80 images vs Hatchback's 50).
5. **Longer fine-tuning** — Unfreeze more layers with a very low learning rate for further gains.